# Complex + CP

LSTM and Transformer point predictors wrapped with split-conformal, ACI, and AgACI.  Tests whether base-model complexity helps interval quality once the uncertainty layer is fixed.

**Workflow.** This notebook (i) makes sure the relevant interval CSVs in `results/` exist (running `scripts/run_grid.py --only complex_cp` if they don't), (ii) loads every CSV and writes a per-cell timeline figure under `report/figures/timelines/Complex_CP/`, (iii) prints a per-cell metrics table and a summary aggregated across (ticker, regime).

**Re-running.** Delete the CSVs in `results/` to force a re-run.

In [ ]:
# sys.path bootstrap so 'from src...' works no matter where the kernel was launched from
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [ ]:
import warnings, glob, subprocess, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.analysis._shared import RESULTS_DIR, CRISIS_WINDOWS, MODEL_CATEGORIES
from src.plotting import plot_interval_timeline, compute_cell_metrics, render_cell_grid


## 1. Run the experiment slice `--only complex_cp` (idempotent)

In [ ]:
# Idempotent: only invokes scripts/run_grid.py if the relevant CSVs are missing.
needed = {
    "lstm_split_AIG_2008.csv",
    "lstm_split_GNW_2008.csv",
    "lstm_split_MGM_2008.csv",
    "lstm_split_MCD_2020.csv",
    "lstm_split_MGM_2020.csv",
    "lstm_split_AZO_2020.csv",
    "lstm_split_SNAP_2022.csv",
    "lstm_split_NFLX_2022.csv",
    "lstm_split_META_2022.csv",
    "lstm_aci_AIG_2008.csv",
    "lstm_aci_GNW_2008.csv",
    "lstm_aci_MGM_2008.csv",
    "lstm_aci_MCD_2020.csv",
    "lstm_aci_MGM_2020.csv",
    "lstm_aci_AZO_2020.csv",
    "lstm_aci_SNAP_2022.csv",
    "lstm_aci_NFLX_2022.csv",
    "lstm_aci_META_2022.csv",
    "lstm_agaci_AIG_2008.csv",
    "lstm_agaci_GNW_2008.csv",
    "lstm_agaci_MGM_2008.csv",
    "lstm_agaci_MCD_2020.csv",
    "lstm_agaci_MGM_2020.csv",
    "lstm_agaci_AZO_2020.csv",
    "lstm_agaci_SNAP_2022.csv",
    "lstm_agaci_NFLX_2022.csv",
    "lstm_agaci_META_2022.csv",
    "transformer_split_AIG_2008.csv",
    "transformer_split_GNW_2008.csv",
    "transformer_split_MGM_2008.csv",
    "transformer_split_MCD_2020.csv",
    "transformer_split_MGM_2020.csv",
    "transformer_split_AZO_2020.csv",
    "transformer_split_SNAP_2022.csv",
    "transformer_split_NFLX_2022.csv",
    "transformer_split_META_2022.csv",
    "transformer_aci_AIG_2008.csv",
    "transformer_aci_GNW_2008.csv",
    "transformer_aci_MGM_2008.csv",
    "transformer_aci_MCD_2020.csv",
    "transformer_aci_MGM_2020.csv",
    "transformer_aci_AZO_2020.csv",
    "transformer_aci_SNAP_2022.csv",
    "transformer_aci_NFLX_2022.csv",
    "transformer_aci_META_2022.csv",
    "transformer_agaci_AIG_2008.csv",
    "transformer_agaci_GNW_2008.csv",
    "transformer_agaci_MGM_2008.csv",
    "transformer_agaci_MCD_2020.csv",
    "transformer_agaci_MGM_2020.csv",
    "transformer_agaci_AZO_2020.csv",
    "transformer_agaci_SNAP_2022.csv",
    "transformer_agaci_NFLX_2022.csv",
    "transformer_agaci_META_2022.csv"
}
have = {p.name for p in RESULTS_DIR.glob("*.csv")} if RESULTS_DIR.exists() else set()
missing = sorted(n for n in needed if n not in have)
if missing:
    print(f"Running scripts/run_grid.py --only complex_cp  ({len(missing)} cells missing)")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "run_grid.py"), "--only", "complex_cp"], check=True)
else:
    print(f"All complex_cp CSVs present in results/, skipping run.")


## 2. Render per-cell timeline figures and per-cell metrics

In [ ]:
# Load every interval CSV produced by this category, render one timeline figure
# per (model, wrapper, ticker, regime) cell, write to disk, return per-cell metrics.
out_dir = ROOT / "report" / "figures" / "timelines" / "Complex_CP"
patterns = []
for m in ['lstm', 'transformer']:
    patterns.append(str(RESULTS_DIR / f"{m}_*.csv"))
files = []
for p in patterns:
    files.extend(glob.glob(p))
files = sorted(files)
if not files:
    raise RuntimeError(f"No CSVs found for category 'Complex_CP'; did the runner finish?")
raw = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in files], ignore_index=True)
metrics = render_cell_grid(
    raw, model_filter=['lstm', 'transformer'], out_dir=out_dir,
    crisis_windows=CRISIS_WINDOWS, category_lookup=MODEL_CATEGORIES,
)
print(f"wrote {len(metrics)} timeline figures → {out_dir.relative_to(ROOT)}")
metrics_sorted = metrics.sort_values(["model", "wrapper", "regime", "ticker"]).reset_index(drop=True)
display(metrics_sorted.round(3))


## 3. Summary aggregated across (ticker, regime)

In [ ]:
# Aggregate by (model, wrapper) — overall mean across all (ticker, regime) cells
summary = (
    metrics_sorted
    .groupby(["model", "wrapper"])[["cov", "winkler", "width"]]
    .mean()
    .sort_values("winkler")
    .round(3)
)
display(summary)

# Show a couple of representative figures inline
sample_files = sorted((ROOT / "report" / "figures" / "timelines").rglob("*.png"))
if sample_files:
    print(f"\nShowing first 3 of {len(sample_files)} timeline figures:")
    for p in sample_files[:3]:
        display(Image(filename=str(p)))
